# Introducción a Machine Learning con Python #

---

Visual Studio Code y la extensión de Python ofrecen un excelente editor para escenarios de ciencia de datos. Gracias a la compatibilidad nativa con Jupyter Notebooks y Anaconda, es fácil comenzar. En esta sección, usaremos un espacio de trabajo para el tutorial, un entorno de **miniconda** con los módulos de ciencia de datos necesarios para crear un modelo de aprendizaje automático.

## 1. Instalar las librerias necesarias. ##

A fecha de julio de 2025 TensorFlow todavia no está soportado para Python 3.13 por lo que hemos de crear un entorno **conda** con una versión de Python inferior si es que no la tenemos ya. Abrir una **Terminal** de Windows y ejecutar el comando:

`conda create -n cursopython3_12 python=3.12`


Después de crear el entorno nos aseguraremos de **Seleccionar el Kernel** de dicho nuevo entorno y a continuación ejecutaremos el siguiente comando para instalar las librerias necesarias:

NOTA: El símbolo **%** significa que es un comando de terminal.

In [1]:
%conda install numpy pandas jupyter seaborn scikit-learn tensorflow keras

3 channel Terms of Service accepted
Channels:
 - defaults
Platform: win-64
Solving environment: done

## Package Plan ##

  environment location: c:\Users\Propietario\miniconda3\envs\cursopython3_12

  added / updated specs:
    - jupyter
    - keras
    - numpy
    - pandas
    - scikit-learn
    - seaborn
    - tensorflow


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libcurl-8.14.1             |       h2300eb9_1         401 KB
    libkrb5-1.21.3             |       hcb72d8e_3         1.2 MB
    libpq-17.4                 |       h4a159e6_2         4.2 MB
    python-3.12.3              |       h1d929f7_0        16.4 MB
    qtbase-6.7.3               |       hd088775_4        20.8 MB
    ------------------------------------------------------------
                                           Total:        43.0 MB

The following NEW packages will be INSTALLED:

  absl-py            pkgs

C:\Users\Propietario\miniconda3\Lib\site-packages\menuinst\platforms\win.py:69: UserWarning: Quick launch menus are not available for system level installs
  warnings.warn("Quick launch menus are not available for system level installs")
Overwriting existing link at C:\ProgramData\Microsoft\Windows\Start Menu\Programs\Anaconda (miniconda3)\Jupyter Notebook (cursopython3_12).lnk.

Terminal profiles are not available for system level installs



Si prefieres ejecutar el comando anterior desde la **Terminal** integrada de __VS Code__, abre la terminal desde el menú __Ver->Terminal__ y ejecuta el comando asegurándote de que activas el entorno ejecutando primero:

`conda activate cursopython3_12`

## 2. Importar los datos. ##

Vamos a utilizar el fichero **titanic3.csv**

In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv('titanic3.csv')
data.head()

,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.00,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.92,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.00,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.00,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


Ahora visualizaremos algunas estadísticas

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.style.use('default')  # Cambiar a 'dark_background' si se prefiere un fondo oscuro
fig, axs = plt.subplots(ncols=5, figsize=(30,5))
sns.violinplot(x="survived", y="age", hue="sex", data=data, ax=axs[0])
sns.pointplot(x="sibsp", y="survived", hue="sex", data=data, ax=axs[1])
sns.pointplot(x="parch", y="survived", hue="sex", data=data, ax=axs[2])
sns.pointplot(x="pclass", y="survived", hue="sex", data=data, ax=axs[3])
sns.violinplot(x="survived", y="fare", hue="sex", data=data, ax=axs[4])

Estos gráficos son útiles para visualizar algunas de las relaciones entre la supervivencia y las variables de entrada de los datos, pero también es posible usar **pandas** para calcular correlaciones. Para ello, todas las variables utilizadas deben ser numéricas para el cálculo de la correlación, y actualmente el género se almacena como una cadena. Para convertir esos valores de cadena a enteros, agregue y ejecute el siguiente código:

In [3]:
data.replace({'male': 1, 'female': 0}, inplace=True)


C:\Users\Propietario\AppData\Local\Temp\ipykernel_15536\761646214.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({'male': 1, 'female': 0}, inplace=True)


Ahora, puede analizar la correlación entre todas las variables de entrada para identificar las características que serían las mejores entradas para un modelo de aprendizaje automático. Cuanto más cercano esté un valor a 1, mayor será la correlación entre el valor y el resultado. Utilice el siguiente código para correlacionar la relación entre todas las variables y la supervivencia.

In [4]:
data.corr(numeric_only=True).abs()[["survived"]]

,survived
pclass,0.312469
survived,1.000000
sex,0.528693
age,0.055512
sibsp,0.027825
parch,0.082660
fare,0.244265
body,NaN


Al observar los resultados de la correlación, notará que algunas variables, como el género, tienen una correlación bastante alta con la supervivencia, mientras que otras, como los parientes (sibsp = hermanos o cónyuge, parch = padres o hijos), parecen tener poca correlación.

Supongamos que sibsp y parch están relacionados en cuanto a su efecto sobre la supervivencia, y agrupémoslos en una nueva columna llamada "parientes" para ver si su combinación tiene una mayor correlación con la supervivencia. Para ello, comprobaremos si, para un pasajero dado, el número de sibsp y parch es mayor que 0 y, de ser así, podemos afirmar que tenía un familiar a bordo.

Utilice el siguiente código para crear una nueva variable y columna en el conjunto de datos llamado relativesy verificar la correlación nuevamente.

In [5]:
data['relatives'] = data.apply (lambda row: int((row['sibsp'] + row['parch']) > 0), axis=1)
data.corr(numeric_only=True).abs()[["survived"]]

,survived
pclass,0.312469
survived,1.000000
sex,0.528693
age,0.055512
sibsp,0.027825
parch,0.082660
fare,0.244265
body,NaN
relatives,0.201719


Observará que, de hecho, al analizar si una persona tenía parientes, en comparación con cuántos, existe una mayor correlación con la supervivencia. Con esta información, puede eliminar del conjunto de datos las columnas sibsp y parch de bajo valor , así como las filas con valores NaN , para obtener un conjunto de datos que pueda usarse para entrenar un modelo.

Aunque la edad tenía una correlación directa baja, se mantuvo porque parece razonable que aún pudiera tener correlación junto con otras entradas.

In [6]:
data = data[['sex', 'pclass','age','relatives','fare','survived']].dropna()

## 3. Entrenar y Evaluar un Modelo. ##

Con el conjunto de datos listo, podemos empezar a crear un modelo. En esta sección, usaremos la biblioteca **scikit-learn** (ya que ofrece funciones auxiliares útiles) para preprocesar el conjunto de datos, entrenar un modelo de clasificación para determinar la supervivencia del Titanic y, a continuación, usar ese modelo con datos de prueba para determinar su precisión.

Un primer paso común para entrenar un modelo es dividir el conjunto de datos en datos de entrenamiento y de validación. Esto permite usar una parte de los datos para entrenar el modelo y otra para probarlo. Si se usaran todos los datos para entrenar el modelo, no habría forma de estimar su rendimiento real con datos que el modelo aún no ha analizado. Una ventaja de la biblioteca **scikit-learn** es que proporciona un método específico para dividir un conjunto de datos en datos de entrenamiento y de prueba.

In [7]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(data[['sex','pclass','age','relatives','fare']], data.survived, test_size=0.2, random_state=0)

A continuación, normalizaremos las entradas para que todas las características se traten por igual. Por ejemplo, dentro del conjunto de datos, los valores de edad oscilan entre ~0 y 100, mientras que el género solo es 1 o 0. Al normalizar todas las variables, puede garantizar que los rangos de valores sean iguales. Utilice el siguiente código para escalar los valores de entrada.

In [8]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train = sc.fit_transform(x_train)
X_test = sc.transform(x_test)

Existen muchos algoritmos de aprendizaje automático diferentes para modelar los datos. La biblioteca scikit-learn también ofrece compatibilidad con muchos de ellos y una tabla para ayudarte a seleccionar el más adecuado para tu situación. Por ahora, utiliza el algoritmo Naïve Bayes , un algoritmo común para problemas de clasificación. Agrega una celda con el siguiente código para crear y entrenar el algoritmo.

In [9]:
from sklearn.naive_bayes import GaussianNB

model = GaussianNB()
model.fit(X_train, y_train)

,priors,None
,var_smoothing,1e-09


Con un modelo entrenado, ahora podemos probarlo con el conjunto de datos de prueba que no se entrenó. Ejecutaremos el siguiente código para predecir el resultado de los datos de prueba y calcular la precisión del modelo.

In [10]:
from sklearn import metrics

predict_test = model.predict(X_test)
print(metrics.accuracy_score(y_test, predict_test))

prediction = model.predict(sc.transform([[0, 1, 22, 0, 7.25]]))  # Ejemplo de predicción
print("Predicción para un pasajero femenino de clase 1, 22 años, sin familiares a bordo y tarifa 7.25:", "Sobrevivió" if prediction[0] == 1 else "No sobrevivió")

0.7464114832535885
Predicción para un pasajero femenino de clase 1, 22 años, sin familiares a bordo y tarifa 7.25: Sobrevivió


c:\Users\Propietario\miniconda3\envs\cursopython3_12\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 4. Utilizar una Red Neuronal ##

Una red neuronal es un modelo que utiliza ponderaciones y funciones de activación, modelando aspectos de las neuronas humanas, para determinar un resultado basado en las entradas proporcionadas. A diferencia del algoritmo de aprendizaje automático que se analizó anteriormente, las redes neuronales son una forma de aprendizaje profundo en la que no es necesario conocer de antemano un algoritmo ideal para el conjunto de problemas. Se puede utilizar en diversos escenarios, y la clasificación es uno de ellos. En esta sección, se utilizará la biblioteca Keras con TensorFlow para construir la red neuronal y explorar cómo gestiona el conjunto de datos del Titanic.

In [12]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()

Tras definir el modelo, el siguiente paso es añadir las capas de la red neuronal. Por ahora, simplifiquemos el proceso y usemos solo tres capas.

In [13]:
model.add(Dense(5, kernel_initializer = 'uniform', activation = 'relu', input_dim = 5))
model.add(Dense(5, kernel_initializer = 'uniform', activation = 'relu'))
model.add(Dense(1, kernel_initializer = 'uniform', activation = 'sigmoid'))

c:\Users\Propietario\miniconda3\envs\cursopython3_12\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


La primera capa se configurará para que tenga una dimensión de 5, ya que tiene cinco entradas: sexo, clase, edad, parientes y tarifa.

La última capa debe generar 1, ya que se desea una salida unidimensional que indique si un pasajero sobreviviría.

La capa intermedia se mantuvo en 5 para simplificar, aunque ese valor podría haber sido diferente.

La función de activación de la unidad lineal rectificada (relu) se utiliza como una buena función de activación general para las primeras dos capas, mientras que la función de activación sigmoidea es necesaria para la capa final, ya que el resultado deseado (si un pasajero sobrevive o no) debe escalarse en el rango de 0 a 1 (la probabilidad de que un pasajero sobreviva).

PAra ver un resumen del modelo:

In [14]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 5)              │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │            30 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             6 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 66 (264.00 B)

 Trainable params: 66 (264.00 B)

 Non-trainable params: 0 (0.00 B)

Una vez creado el modelo, es necesario compilarlo. Para ello, es necesario definir el tipo de optimizador que se utilizará, cómo se calculará la pérdida y qué métrica se optimizará. Agregue el siguiente código para compilar y entrenar el modelo.

In [15]:
model.compile(optimizer="adam", loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, batch_size=32, epochs=50)

Epoch 1/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.4952 - loss: 0.6931
Epoch 2/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6011 - loss: 0.6913
Epoch 3/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6012 - loss: 0.6893
Epoch 4/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5731 - loss: 0.6883
Epoch 5/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5959 - loss: 0.6840
Epoch 6/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6419 - loss: 0.6795
Epoch 7/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7403 - loss: 0.6729
Epoch 8/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7599 - loss: 0.6622
Epoch 9/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7477 - loss: 0.6491
Epoch 10/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7648 - loss: 0.6300
Epoch 11/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7527 - loss: 0.6022
Epoch 12/50
27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7621 - lo

Ahora que el modelo está construido y entrenado, podemos ver cómo funciona con los datos de prueba.

In [16]:
y_pred = np.rint(model.predict(X_test).flatten())
print(metrics.accuracy_score(y_test, y_pred))

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
0.7942583732057417


Con esta sencilla red neuronal, el resultado es mejor que el 75 % de precisión del clasificador Naive Bayes probado anteriormente.

Si lo que queremos es calcular un resultado nuevo, podemos hacer lo siguiente:

In [20]:
My_Test= np.array([[0, 1, 22, 0, 7.25]])
My_Test = sc.transform(My_Test)

y_pred = np.rint(model.predict(My_Test).flatten())
print("Probabilidad de supervivencia:", y_pred[0])

c:\Users\Propietario\miniconda3\envs\cursopython3_12\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
Probabilidad de supervivencia: 1.0
